In [1]:
import re
import sys
import math
from pathlib import Path 

import pandas as pd
import sqlalchemy as db
import plotly.express as px
import plotly.graph_objects as go
from huggingface_hub import hf_hub_download
from dotenv import find_dotenv, load_dotenv

In [2]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [3]:
# 1. Load variables (makes 'import utils' work via PYTHONPATH=.)
load_dotenv(find_dotenv())

# 2. Define the project root based on where the .env file lives
ROOT_DIR = Path(find_dotenv()).parent

# 3. Explicitly anchor the notebook folder to the root 
# (Change this string depending on which subfolder the notebook lives in)
NOTEBOOK_DIR = ROOT_DIR / "research" / "analyses"

In [4]:
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import utils

In [5]:
def time_to_seconds(time_str):
    if not time_str:
        return None
    # Match optional hours and optional minutes using regular expressions
    # \d+ looks for one or more digits
    hours_match = re.search(r'(\d+)\s*h', time_str)
    minutes_match = re.search(r'(\d+)\s*m', time_str)
    
    # Extract the numbers if found, otherwise default to 0
    hours = int(hours_match.group(1)) if hours_match else 0
    minutes = int(minutes_match.group(1)) if minutes_match else 0
    
    # Calculate total seconds
    total_seconds = (hours * 3600) + (minutes * 60)
    return total_seconds

In [6]:
layout = {
    'title': {
        'text': '',
        'font': {'size': 22, 'family': 'Raleway', 'color': 'white'},
        'x': 0.5,
        'y': 0.95,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    'xaxis': {
        'title': {
            'text': '',
            'font': {'size': 18, 'family': 'Raleway', 'color': 'white'} 
        },
        'tickfont': {'size': 14, 'family': 'Roboto', 'color': 'white'}
    },
    'yaxis': {
        'layer': 'below traces',
        'title': {
            'text': '',
            'font': {'size': 18, 'family': 'Raleway', 'color': 'white'} 
        },
        'tickfont': {'size': 14, 'family': 'Roboto', 'color': 'white'}
    },
    'font': {'color': 'white'},
    'paper_bgcolor': '#1A1A2E',
    'plot_bgcolor': 'rgba(61, 61, 61, 0)'
}

# Load IMDb Dataset

In [7]:
dw_local_path = hf_hub_download(
    repo_id="astaileyyoung/CineStat",
    filename="widescreen_sample.parquet",
    repo_type="dataset",
    local_dir="./data/"
)

In [8]:
population = pd.read_parquet("./data/widescreen_sample.parquet")
population = population[population['is_major'] == 1]
population

,tconst,titleType,primaryTitle,originalTitle,isAdult,year,endYear,runtimeMinutes,genres,imdb_id,aspect_ratio,printed_formats,colorations,production_companies,countries,rating,processes,distribution_companies,is_major
11,tt0039213,movie,Borrowed Trouble,Borrowed Trouble,0,1948,None,58.0,"Drama,Western",39213,"1.37 : 1,",35 mm,Black and White,Hopalong Cassidy Productions Inc.,United States,6.3,Spherical,United Artists|United Artists|United Artists|U...,1
13,tt0039248,movie,El casado casa quiere,El casado casa quiere,0,1948,None,83.0,"Action,Comedy,Drama",39248,"1.37 : 1,",35 mm,Black and White,Ramex Films,Mexico,5.6,Spherical,RKO Radio Pictures de México|Clasa-Mohme|RKO R...,1
20,tt0039304,movie,Daybreak,Daybreak,0,1948,None,75.0,Drama,39304,"1.37 : 1,",35 mm,Black and White,Sydney Box Productions,United Kingdom,6.7,Spherical,General Film Distributors (GFD)|Kommunenes Fil...,1
28,tt0039550,movie,Larceny,Larceny,0,1948,None,89.0,"Crime,Drama,Film-Noir",39550,"1.37 : 1,",35 mm,Black and White,Universal International Pictures (UI),United States,6.8,Spherical,Universal Pictures|Empire Universal Films|Gene...,1
29,tt0039613,movie,Mary Lou,Mary Lou,0,1948,None,65.0,"Music,Romance",39613,"1.37 : 1,",35 mm,Black and White,Sam Katzman Productions,United States,NaN,Spherical,Columbia Pictures|Columbia Pictures of Canada|...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7280,tt0053935,movie,Ice Cold in Alex,Ice Cold in Alex,0,1958,None,130.0,"Adventure,Drama,War",53935,"1.66 : 1,",35 mm,Black and White|Black and White,Associated British Picture Corporation (ABPC),United Kingdom,7.7,Spherical,Associated British-Pathé|Lehmacher Filmverleih...,1
7282,tt0054192,movie,The Poacher's Daughter,Sally's Irish Rogue,0,1958,None,74.0,Comedy,54192,"1.37 : 1,",35 mm,Black and White,Emmett Dalton Productions,United Kingdom,4.9,Spherical,British Lion Film Corporation|RKO Radio Pictur...,1
7290,tt0122119,movie,Island Women,Island Women,0,1958,None,72.0,Drama,122119,"1.37 : 1,",35 mm,Black and White,Security Pictures,United States,6.8,,United Artists|United Artists,1
7294,tt0135834,movie,Cerf-volant du bout du monde,Cerf-volant du bout du monde,0,1958,None,82.0,"Adventure,Family,Fantasy",135834,"1.37 : 1,",35 mm,Color,Garance Productions|Les Films Jean Tourane|Coc...,France|China,6.8,Spherical,Cocinor|Xerox Films|Paramount Pictures|Tamasa ...,1


In [9]:
population_g = population.groupby("year").count()
population_g = population_g.reset_index()
population_g

,year,tconst,titleType,primaryTitle,originalTitle,isAdult,endYear,runtimeMinutes,genres,imdb_id,aspect_ratio,printed_formats,colorations,production_companies,countries,rating,processes,distribution_companies,is_major
0,1948,267,267,267,267,267,0,267,267,267,267,267,267,267,267,259,267,267,267
1,1949,262,262,262,262,262,0,262,262,262,262,262,262,262,262,261,262,262,262
2,1950,278,278,278,278,278,0,278,278,278,278,278,278,278,278,276,278,278,278
3,1951,298,298,298,298,298,0,298,298,298,298,298,298,298,298,297,298,298,298
4,1952,282,282,282,282,282,0,282,282,282,282,282,282,282,282,282,282,282,282
5,1953,304,304,304,304,304,0,302,304,304,304,304,304,304,304,292,304,304,304
6,1954,217,217,217,217,217,0,216,216,217,217,217,217,217,217,216,217,217,217
7,1955,216,216,216,216,216,0,216,216,216,216,216,216,216,216,215,216,216,216
8,1956,233,233,233,233,233,0,233,233,233,233,233,233,233,233,233,233,233,233
9,1957,279,279,279,279,279,0,279,279,279,279,279,279,279,279,279,279,279,279


In [10]:
fig = px.bar(population_g, x='year', y='tconst')
fig.update_layout(layout)
fig.update_layout(title=dict(
    text="Total Films by Year"
))
fig.update_traces(marker_color="#3bbdb6")

# Get Existing Films

In [11]:
dw_local_path = hf_hub_download(
    repo_id="astaileyyoung/CineStat",
    filename="CineStat_agg.duckdb",
    repo_type="dataset",
    local_dir=str(ROOT_DIR)
)

In [12]:
db_url = f"duckdb:///{ROOT_DIR / 'CineStat_agg.duckdb'}"
engine = db.create_engine(db_url, connect_args={"read_only": True})

In [13]:
existing = pd.read_sql("""
    SELECT * 
    FROM dimWork 
    WHERE kind = 'movie' 
        AND year >= 1948
        AND year <= 1958
""", engine)
existing

,work_id,imdb_id,series_imdb_id,title,year,season,episode,release_date,kind,rating,metacritic_rating,votes,budget,gross,runtime,plot,is_major,title_localized
0,34272,33996,None,Panhandle,1948,None,None,1948-02-22,movie,6.3,NaN,429,NaN,NaN,85,"John Sands, a former Texas marshal turns to ra...",0,Panhandle
1,34812,39188,None,Bill and Coo,1948,None,None,1948-03-28,movie,5.6,NaN,342,NaN,NaN,61,The feathered residents of Chirpendale are ter...,0,Bill and Coo
2,34816,39195,None,Blanche Fury,1948,None,None,1948-09-08,movie,6.7,NaN,1259,1500000 USD,NaN,90,The childless widow of Allan Fury bequeaths th...,0,Blanche Fury
3,34823,39220,None,Brighton Rock,1948,None,None,1951-11-07,movie,7.3,NaN,7516,NaN,72464 USD,92,"In Brighton in 1935, small-time gang leader Pi...",0,Brighton Rock
4,34836,39304,None,Daybreak,1948,None,None,1949-05-13,movie,6.7,NaN,306,NaN,NaN,75,A hangman conceals his true identity when he f...,1,Daybreak
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1803,58211,50527,None,How to Murder a Rich Uncle,1957,None,None,1957-06-01,movie,6.9,NaN,314,NaN,NaN,79,The impoverished Clitterburn family live on a ...,1,How to Murder a Rich Uncle
1804,58212,50622,None,The Land Unknown,1957,None,None,1957-10-15,movie,5.7,NaN,2228,NaN,NaN,78,Three men and a woman crash-land in a deep cra...,1,The Land Unknown
1805,58213,51141,None,Until They Sail,1957,None,None,1957-10-23,movie,6.5,NaN,1872,1841000 USD,NaN,94,"During WWII, unmarried New Zealand women meet ...",1,Until They Sail
1806,58214,51913,None,The Matchmaker,1958,None,None,1959-08-14,movie,6.8,NaN,1290,NaN,NaN,103,Thornton Wilder's tale of a matchmaker who des...,1,The Matchmaker


In [14]:
minors = existing[existing['is_major'] == 0]
minors.shape[0]

567

In [15]:
sample = existing[existing['is_major'] == 1]
sample.shape[0]

1241

In [16]:
sample_g = sample.groupby('year').count()
sample_g = sample_g.reset_index(drop=False)
sample_g

,year,work_id,imdb_id,series_imdb_id,title,season,episode,release_date,kind,rating,metacritic_rating,votes,budget,gross,runtime,plot,is_major,title_localized
0,1948,100,100,0,100,0,0,100,100,100,10,100,37,12,100,100,100,100
1,1949,109,109,0,109,0,0,109,109,109,9,109,36,10,109,109,109,109
2,1950,109,109,0,109,0,0,109,109,109,14,109,29,18,109,109,109,109
3,1951,110,110,0,110,0,0,110,110,110,11,110,31,8,110,110,110,110
4,1952,101,101,0,101,0,0,101,101,101,9,101,22,14,101,101,101,101
5,1953,98,98,0,98,0,0,98,98,98,11,98,40,14,98,98,98,98
6,1954,97,97,0,97,0,0,97,97,97,12,97,37,22,97,97,97,97
7,1955,131,131,0,131,0,0,131,131,131,14,131,49,21,131,131,131,131
8,1956,126,126,0,126,0,0,126,126,126,15,126,46,24,126,126,126,126
9,1957,146,146,0,146,0,0,146,146,146,18,146,49,25,146,146,146,146


In [17]:
fig = px.bar(sample_g, x='year', y='work_id')
fig.update_layout(layout)
fig.update_layout(title=dict(
    text="Sample Films by Year"
))
fig.update_traces(marker_color="#3bbdb6")

# Calculate Sample Size

In [18]:
utils.calculate_moe(866, 2877)

0.027847019832817545

In [19]:
# yearly_data = {}
# for x in sample_g['year']:
#     try:
#         sample_total = sample_g[sample_g['year'] == x]['work_id'].iloc[0]
#         population_total = population_g[population_g['year'] == x]['imdb_id'].iloc[0]
#         yearly_data[x] = utils.calculate_moe(sample_total, population_total)
#     except Exception as e:
#         print(x, e)
# yearly_data

In [20]:
yearly_data = {}
for x in sample_g['year']:
    try:
        yearly_data[x] = (sample_g[sample_g['year'] == x]['work_id'].iloc[0], population_g[population_g['year'] == x]['imdb_id'].iloc[0])
    except Exception as e:
        print(x, e)
yearly_data

{1948: (np.int64(100), np.int64(267)),
 1949: (np.int64(109), np.int64(262)),
 1950: (np.int64(109), np.int64(278)),
 1951: (np.int64(110), np.int64(298)),
 1952: (np.int64(101), np.int64(282)),
 1953: (np.int64(98), np.int64(304)),
 1954: (np.int64(97), np.int64(217)),
 1955: (np.int64(131), np.int64(216)),
 1956: (np.int64(126), np.int64(233)),
 1957: (np.int64(146), np.int64(279)),
 1958: (np.int64(114), np.int64(241))}

In [21]:
print(f"{'Year':<6} | {'Population (N)':<15} | {'Sample (n)':<10} | {'Sampling Rate':<15} | {'Margin of Error':<15}")
print("-" * 65)

data = []
for year, (n, N) in yearly_data.items():
    if n > N:
        n = N
    moe = utils.calculate_moe(n, N)
    rate = (n / N) * 100
    print(f"{year:<6} | {N:<15} | {n:<10} | {rate:>12.1f}% | {moe * 100:>13.2f}%")
    d = {
        "year": year,
        "rate": rate,
        "population": N,
        "sample": n,
        "moe": moe
    }
    data.append(d)
moe_df = pd.DataFrame(data)
moe_df.round(3)

Year   | Population (N)  | Sample (n) | Sampling Rate   | Margin of Error
-----------------------------------------------------------------
1948   | 267             | 100        |         37.5% |          7.77%
1949   | 262             | 109        |         41.6% |          7.19%
1950   | 278             | 109        |         39.2% |          7.33%
1951   | 298             | 110        |         36.9% |          7.43%
1952   | 282             | 101        |         35.8% |          7.83%
1953   | 304             | 98         |         32.2% |          8.16%
1954   | 217             | 97         |         44.7% |          7.42%
1955   | 216             | 131        |         60.6% |          5.38%
1956   | 233             | 126        |         54.1% |          5.93%
1957   | 279             | 146        |         52.3% |          5.61%
1958   | 241             | 114        |         47.3% |          6.68%


,year,rate,population,sample,moe
0,1948,37.453,267,100,0.078
1,1949,41.603,262,109,0.072
2,1950,39.209,278,109,0.073
3,1951,36.913,298,110,0.074
4,1952,35.816,282,101,0.078
5,1953,32.237,304,98,0.082
6,1954,44.700,217,97,0.074
7,1955,60.648,216,131,0.054
8,1956,54.077,233,126,0.059
9,1957,52.330,279,146,0.056


In [22]:
def calculate_required_sample(N, target_moe=0.03):
    z = 1.96  # 95% confidence
    p = 0.5   # Worst-case variance for maximum conservative target
    
    # Baseline sample size for an infinite population
    n_0 = (z**2 * p * (1 - p)) / (target_moe**2)
    
    # Apply Finite Population Correction backwards to find required n for this population
    n = n_0 / (1 + ((n_0 - 1) / N))
    
    return math.ceil(n)

In [23]:
print(f"{'Year':<6} | {'Population (N)':<15} | {'Required Sample (n)':<20} | {'Required Rate':<15}")
print("-" * 65)

for year, (current_n, N) in yearly_data.items():
    # Calculates required sample for a +/-3% margin of error (0.03)
    required_n = calculate_required_sample(N, target_moe=0.03)
    rate = (required_n / N) * 100
    
    print(f"{year:<6} | {N:<15} | {required_n:<20} | {rate:>12.1f}%")

Year   | Population (N)  | Required Sample (n)  | Required Rate  
-----------------------------------------------------------------
1948   | 267             | 214                  |         80.1%
1949   | 262             | 211                  |         80.5%
1950   | 278             | 221                  |         79.5%
1951   | 298             | 234                  |         78.5%
1952   | 282             | 224                  |         79.4%
1953   | 304             | 237                  |         78.0%
1954   | 217             | 181                  |         83.4%
1955   | 216             | 180                  |         83.3%
1956   | 233             | 192                  |         82.4%
1957   | 279             | 222                  |         79.6%
1958   | 241             | 197                  |         81.7%


In [24]:
not_in_population = sample[~sample['imdb_id'].isin(population['imdb_id'])]
not_in_population

,work_id,imdb_id,series_imdb_id,title,year,season,episode,release_date,kind,rating,metacritic_rating,votes,budget,gross,runtime,plot,is_major,title_localized
488,35445,44214,None,White Corridors,1951,None,None,1951-10-01,movie,7.0,NaN,210,NaN,NaN,102,"Hospital drama set at the Yeoman's Hospital, i...",1,White Corridors
665,35628,46181,None,Personal Affair,1953,None,None,1954-01-15,movie,6.5,NaN,664,NaN,NaN,82,"In a 1950s British village, a teenager, who is...",1,Personal Affair
1102,36083,51492,None,Count Five and Die,1957,None,None,1958-03-01,movie,6.5,NaN,247,NaN,NaN,92,American and British counter-espionage combine...,1,Count Five and Die
1346,53778,45681,None,Desperate Moment,1953,None,None,1953-08-01,movie,6.6,NaN,245,NaN,NaN,88,Simon Van Halder (Sir Dirk Bogarde) is accused...,1,Desperate Moment


In [25]:
not_in_sample = population[~population['imdb_id'].isin(sample['imdb_id'])]
not_in_sample[['primaryTitle', 'year', 'distribution_companies', 'runtimeMinutes']].sort_values(by='runtimeMinutes', ascending=False)

,primaryTitle,year,distribution_companies,runtimeMinutes
6458,Quiet Flows the Don,1957,Gala Film Distributors|Deutsche Film Hansa|Tel...,330.0
4200,Gunfighters of the Northwest,1954,Columbia Pictures|Columbia Pictures of Canada|...,315.0
2068,Captain Video: Master of the Stratosphere,1951,Columbia Pictures|Columbia Pictures of Canada|...,287.0
4344,Riding with Buffalo Bill,1954,Columbia Pictures|Columbia Pictures Proprietar...,280.0
441,Tex Granger: Midnight Rider of the Plains,1948,Columbia Pictures|Columbia Pictures of Canada|...,270.0
...,...,...,...,...
3884,4th of July Firecrackers,1953,RKO Radio Pictures,45.0
5284,Rocky Marciano vs. Archie Moore,1955,Theater Network Television|United Artists,19.0
3956,Mr. Magoo Cartoon Merry-Go-Round,1953,Columbia Pictures,NaN
3957,Short Subject Star Parade,1953,Columbia Pictures,NaN


# By Year

In [26]:
query = """
    SELECT fw.*, dw.year
    FROM dimWork dw
    INNER JOIN factWork fw ON dw.work_id = fw.work_id
    WHERE dw.year >= 1948
        AND dw.year <= 1958
        AND is_major = 1
"""
sample_data = pd.read_sql(query, engine)
sample_data

,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,min_size,...,z_v_density_tv_g,z_pct_face_tv,z_pct_face_tv_g,z_v_pct_face_tv,z_v_pct_face_tv_g,z_pct_top1_tv,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g,year
0,2870,39304,34836,0.0279,0.0636,0.0347,0.0636,0.2347,0.1444,0.0005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1948
1,2924,39550,34887,0.0313,0.0618,0.0414,0.0669,0.2702,0.2685,0.0003,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1948
2,2997,40002,34945,0.0257,0.0620,0.0353,0.0620,0.2157,0.2157,0.0005,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1948
3,2999,40064,34947,0.0108,0.0329,0.0148,0.0334,0.1681,0.1443,0.0003,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1948
4,3000,40087,34948,0.0315,0.0649,0.0418,0.0650,0.2626,0.2600,0.0006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1948
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1239,7983,127562,52423,0.0172,0.0660,0.0264,0.0660,0.2337,0.1404,0.0004,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1957
1240,8130,169625,55409,0.0324,0.0879,0.0387,0.0879,0.3009,0.0879,0.0016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1952
1241,8163,174866,38855,0.0156,0.0677,0.0192,0.0679,0.2666,0.2666,0.0007,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1948
1242,8179,178372,38867,0.0166,0.0613,0.0229,0.0613,0.1628,0.1355,0.0006,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1950


In [27]:
sample_data_g = sample_data.groupby('year').mean()
sample_data_g = sample_data_g.reset_index()
sample_data_g

,year,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,...,z_v_density_tv,z_v_density_tv_g,z_pct_face_tv,z_pct_face_tv_g,z_v_pct_face_tv,z_v_pct_face_tv_g,z_pct_top1_tv,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g
0,1948,3164.600000,41912.630000,39155.810000,0.021249,0.060206,0.029446,0.061446,0.264748,0.251190,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1949,3204.348624,41582.220183,39450.724771,0.022212,0.061074,0.029999,0.062185,0.260824,0.237186,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950,3437.572727,43881.354545,40844.245455,0.022457,0.063831,0.031105,0.065205,0.275249,0.251546,...,0.0,0.0,0.0,-0.493197,0.0,-0.462805,0.0,0.002793,0.0,0.955955
3,1951,3591.145455,44434.190909,39260.318182,0.022334,0.063085,0.030569,0.064086,0.263026,0.232192,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1952,3785.117647,46020.656863,41201.343137,0.021182,0.063687,0.029558,0.064650,0.268384,0.237183,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1953,3799.292929,46000.313131,40664.767677,0.021037,0.066910,0.029911,0.068095,0.274136,0.249869,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1954,3989.731959,47147.134021,43900.814433,0.017331,0.052302,0.024732,0.053103,0.207749,0.188687,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1955,4175.503817,48345.374046,47536.404580,0.015504,0.046150,0.022082,0.046861,0.174911,0.153668,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1956,4406.666667,49480.158730,47276.857143,0.017210,0.052068,0.024883,0.052935,0.204504,0.170355,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1957,4497.602740,51184.609589,47187.746575,0.019671,0.055034,0.027789,0.056227,0.223649,0.188322,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
sample_data_g = sample_data_g.merge(
    moe_df,
    how='inner',
    on='year'
)
sample_data_g

,year,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,...,z_v_pct_face_tv,z_v_pct_face_tv_g,z_pct_top1_tv,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g,rate,population,sample,moe
0,1948,3164.600000,41912.630000,39155.810000,0.021249,0.060206,0.029446,0.061446,0.264748,0.251190,...,NaN,NaN,NaN,NaN,NaN,NaN,37.453184,267,100,0.077650
1,1949,3204.348624,41582.220183,39450.724771,0.022212,0.061074,0.029999,0.062185,0.260824,0.237186,...,NaN,NaN,NaN,NaN,NaN,NaN,41.603053,262,109,0.071868
2,1950,3437.572727,43881.354545,40844.245455,0.022457,0.063831,0.031105,0.065205,0.275249,0.251546,...,0.0,-0.462805,0.0,0.002793,0.0,0.955955,39.208633,278,109,0.073319
3,1951,3591.145455,44434.190909,39260.318182,0.022334,0.063085,0.030569,0.064086,0.263026,0.232192,...,NaN,NaN,NaN,NaN,NaN,NaN,36.912752,298,110,0.074341
4,1952,3785.117647,46020.656863,41201.343137,0.021182,0.063687,0.029558,0.064650,0.268384,0.237183,...,NaN,NaN,NaN,NaN,NaN,NaN,35.815603,282,101,0.078262
5,1953,3799.292929,46000.313131,40664.767677,0.021037,0.066910,0.029911,0.068095,0.274136,0.249869,...,NaN,NaN,NaN,NaN,NaN,NaN,32.236842,304,98,0.081625
6,1954,3989.731959,47147.134021,43900.814433,0.017331,0.052302,0.024732,0.053103,0.207749,0.188687,...,NaN,NaN,NaN,NaN,NaN,NaN,44.700461,217,97,0.074166
7,1955,4175.503817,48345.374046,47536.404580,0.015504,0.046150,0.022082,0.046861,0.174911,0.153668,...,NaN,NaN,NaN,NaN,NaN,NaN,60.648148,216,131,0.053837
8,1956,4406.666667,49480.158730,47276.857143,0.017210,0.052068,0.024883,0.052935,0.204504,0.170355,...,NaN,NaN,NaN,NaN,NaN,NaN,54.077253,233,126,0.059291
9,1957,4497.602740,51184.609589,47187.746575,0.019671,0.055034,0.027789,0.056227,0.223649,0.188322,...,NaN,NaN,NaN,NaN,NaN,NaN,52.329749,279,146,0.056099


## Avg. Size

In [29]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_size'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Face Size, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Density

In [30]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_f_per_fr'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Faces per Frame, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Distance

In [31]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_dist'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Distance from Center, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Gini

In [32]:
x = sample_data_g['year'].tolist()
y = sample_data_g['gini'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Gini Score, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Dispersion

In [33]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_disp'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Dispersion, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Horizontal Spread

In [34]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_h_spread'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Horizontal Spread, 1948-1958", y=0.9, x=0.5)))
fig.show()

# Aspect Ratio

In [35]:
dw_local_path = hf_hub_download(
    repo_id="astaileyyoung/CineStat",
    filename="imdb.duckdb",
    repo_type="dataset",
    local_dir=str(ROOT_DIR)
)

In [36]:
db_url = f"duckdb:///{ROOT_DIR / 'imdb.duckdb'}"
imdb_engine = db.create_engine(db_url, connect_args={"read_only": True})

In [37]:
query = """
    SELECT 
        tb.tconst,
        tb.primaryTitle,
        tb.startYear, 
        ar.ratio_standardized
    FROM title_basics tb
    INNER JOIN title_aspect_ratio tar ON tb.tconst = tar.tconst
    INNER JOIN aspect_ratio ar ON tar.aspect_ratio_id = ar.id
        AND tb.startYear >= 1948
        AND tb.startYear <= 1958
        AND tb.is_major = 1
"""
pop_aspect = pd.read_sql(query, imdb_engine)
pop_aspect

,tconst,primaryTitle,startYear,ratio_standardized
0,tt12324042,Walt Disney's Hallowe'en Hilarities,1953,1.33
1,tt0039613,Mary Lou,1948,1.33
2,tt0039660,Night Unto Night,1949,1.33
3,tt0039890,The Tender Years,1948,1.33
4,tt0040002,A Woman's Vengeance,1948,1.33
...,...,...,...,...
3153,tt0039550,Larceny,1948,1.33
3154,tt0121515,The Man with My Face,1951,1.33
3155,tt0122119,Island Women,1958,1.33
3156,tt0122292,Tucson,1949,1.33


In [38]:
p_cnt = pop_aspect[['ratio_standardized', 'tconst']].groupby('ratio_standardized').count().rename({'tconst': 'population'}, axis=1)
p_cnt = p_cnt.reset_index()
p_cnt['ratio_standardized'] = p_cnt['ratio_standardized'].round(2)
p_cnt['ratio_standardized'] = p_cnt['ratio_standardized'].astype(str)
p_cnt

,ratio_standardized,population
0,1.33,1990
1,1.66,218
2,1.85,555
3,2.35,248
4,2.55,147


In [39]:
query = """
    SELECT
        fw.*,
        dw.title,
        dw.year,
        dap.ratio,
        dap.ratio_decimal,
        dap.ratio_standardized,
        dap.is_wide,
        dap.is_flat
    FROM factWork fw
    INNER JOIN dimWork dw ON fw.work_id = dw.work_id
    INNER JOIN bridgeAspectRatio bap ON dw.work_id = bap.work_id
    INNER JOIN dimAspectRatio dap ON bap.aspect_ratio_id = dap.aspect_ratio_id
    WHERE dw.year >= 1948
        AND dw.year <= 1958
        AND kind = 'movie'
        AND is_major = 1
"""
aspect = pd.read_sql(query, engine)
aspect['ratio_standardized'] = aspect['ratio_standardized'].round(2)
aspect

,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,min_size,...,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g,title,year,ratio,ratio_decimal,ratio_standardized,is_wide,is_flat
0,2870,39304,34836,0.0279,0.0636,0.0347,0.0636,0.2347,0.1444,0.0005,...,None,None,None,Daybreak,1948,1.37 : 1,1.37,1.33,0,1
1,2924,39550,34887,0.0313,0.0618,0.0414,0.0669,0.2702,0.2685,0.0003,...,None,None,None,Larceny,1948,1.37 : 1,1.37,1.33,0,1
2,2997,40002,34945,0.0257,0.0620,0.0353,0.0620,0.2157,0.2157,0.0005,...,None,None,None,A Woman's Vengeance,1948,1.37 : 1,1.37,1.33,0,1
3,2999,40064,34947,0.0108,0.0329,0.0148,0.0334,0.1681,0.1443,0.0003,...,None,None,None,3 Godfathers,1948,1.37 : 1,1.37,1.33,0,1
4,3000,40087,34948,0.0315,0.0649,0.0418,0.0650,0.2626,0.2600,0.0006,...,None,None,None,All My Sons,1948,1.37 : 1,1.37,1.33,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1381,7943,121515,38635,0.0257,0.0752,0.0339,0.0752,0.3889,0.3889,0.0005,...,None,None,None,The Man with My Face,1951,1.37 : 1,1.37,1.33,0,1
1382,7983,127562,52423,0.0172,0.0660,0.0264,0.0660,0.2337,0.1404,0.0004,...,None,None,None,Four Boys and a Gun,1957,1.37 : 1,1.37,1.33,0,1
1383,8163,174866,38855,0.0156,0.0677,0.0192,0.0679,0.2666,0.2666,0.0007,...,None,None,None,Let's Live Again,1948,1.37 : 1,1.37,1.33,0,1
1384,8179,178372,38867,0.0166,0.0613,0.0229,0.0613,0.1628,0.1355,0.0006,...,None,None,None,"David Harding, Counterspy",1950,1.37 : 1,1.37,1.33,0,1


In [40]:
s_cnt = aspect[['ratio_standardized', 'work_id']].groupby('ratio_standardized').count().rename({'work_id': 'sample'}, axis=1)
s_cnt = s_cnt.reset_index()
s_cnt['ratio_standardized'] = s_cnt['ratio_standardized'].astype(str)
s_cnt

,ratio_standardized,sample
0,1.33,724
1,1.66,90
2,1.85,274
3,2.35,170
4,2.55,128


In [41]:
moe = p_cnt.merge(s_cnt,
                  on='ratio_standardized')
moe['moe'] = moe.apply(lambda x: utils.calculate_moe(x['sample'], x['population']), axis=1)
moe

,ratio_standardized,population,sample,moe
0,1.33,1990,724,0.029057
1,1.66,218,90,0.079338
2,1.85,555,274,0.042165
3,2.35,248,170,0.042238
4,2.55,147,128,0.031248


## Sample by Aspect Ratio

In [42]:
fig = px.bar(s_cnt, x='ratio_standardized', y='sample', color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Count by Aspect Ratio"))
fig.show()

In [43]:
g = aspect[[
    'ratio_standardized',
    'avg_size',
    'std_size',
    'avg_fr_pct_face',
    'std_fr_pct_face',
    'avg_f_per_fr',
    'std_f_per_fr',
    'avg_dist',
    'std_dist',
    'gini',
    'avg_disp',
    'avg_h_spread',
    'std_h_spread',
    'pct_tl',
    'pct_tc',
    'pct_tr',
    'pct_ml',
    'pct_mc',
    'pct_mr',
    'pct_bl',
    'pct_bc',
    'pct_br'
]].groupby('ratio_standardized').mean().round(3)
g = g.reset_index()
g['ratio_standardized'] = g['ratio_standardized'].astype(str)
g

,ratio_standardized,avg_size,std_size,avg_fr_pct_face,std_fr_pct_face,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,...,std_h_spread,pct_tl,pct_tc,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br
0,1.33,0.022,0.026,0.042,0.037,2.027,1.413,0.371,0.145,0.486,...,0.128,0.116,0.276,0.114,0.111,0.250,0.115,0.005,0.006,0.006
1,1.66,0.021,0.025,0.041,0.036,2.112,1.643,0.357,0.147,0.489,...,0.127,0.101,0.246,0.098,0.122,0.289,0.121,0.007,0.008,0.008
2,1.85,0.019,0.024,0.041,0.036,2.237,1.684,0.363,0.150,0.484,...,0.128,0.105,0.237,0.101,0.126,0.282,0.127,0.007,0.008,0.008
3,2.35,0.016,0.018,0.035,0.030,2.313,1.866,0.371,0.153,0.474,...,0.131,0.107,0.229,0.098,0.138,0.273,0.129,0.009,0.009,0.009
4,2.55,0.010,0.011,0.026,0.022,2.682,2.315,0.375,0.154,0.480,...,0.125,0.105,0.221,0.097,0.143,0.271,0.137,0.009,0.008,0.009


In [44]:
g = g.merge(moe.drop('population', axis=1),
            on='ratio_standardized')
g

,ratio_standardized,avg_size,std_size,avg_fr_pct_face,std_fr_pct_face,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,...,pct_tc,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,sample,moe
0,1.33,0.022,0.026,0.042,0.037,2.027,1.413,0.371,0.145,0.486,...,0.276,0.114,0.111,0.250,0.115,0.005,0.006,0.006,724,0.029057
1,1.66,0.021,0.025,0.041,0.036,2.112,1.643,0.357,0.147,0.489,...,0.246,0.098,0.122,0.289,0.121,0.007,0.008,0.008,90,0.079338
2,1.85,0.019,0.024,0.041,0.036,2.237,1.684,0.363,0.150,0.484,...,0.237,0.101,0.126,0.282,0.127,0.007,0.008,0.008,274,0.042165
3,2.35,0.016,0.018,0.035,0.030,2.313,1.866,0.371,0.153,0.474,...,0.229,0.098,0.138,0.273,0.129,0.009,0.009,0.009,170,0.042238
4,2.55,0.010,0.011,0.026,0.022,2.682,2.315,0.375,0.154,0.480,...,0.221,0.097,0.143,0.271,0.137,0.009,0.008,0.009,128,0.031248


## Size

In [45]:
temp = g.copy()
temp['abs_error'] = temp['avg_size'] * g['moe']
fig = px.bar(temp, 
             x='ratio_standardized', 
             y=['avg_size', 'std_size'], 
             barmode='group',
             color_discrete_sequence=["#3bbdb6", "#e97bec"], 
             error_y='abs_error')
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Size by Aspect Ratio"))
fig.update_traces(error_y=dict(
    thickness=2,
    width=6,
    color='white'
))
fig.show()

In [46]:
temp = g.copy()
temp['abs_error'] = temp['avg_fr_pct_face'] * g['moe']
fig = px.bar(temp, x='ratio_standardized', y=['avg_fr_pct_face', 'std_fr_pct_face'], color_discrete_sequence=["#3bbdb6", "#e97bec"], barmode='group', error_y='abs_error')
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Size by Aspect Ratio"))
fig.update_traces(error_y=dict(
    thickness=2,
    width=6,
    color='white'
))
fig.show()

## Density

In [47]:
temp = g.copy()
temp['abs_error'] = temp['avg_f_per_fr'] * temp['moe']
fig = px.bar(temp, x='ratio_standardized', y=['avg_f_per_fr', 'std_f_per_fr'], error_y='abs_error', barmode='group', color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Faces per Frame by Aspect Ratio"))
fig.update_traces(error_y=dict(
    thickness=2,
    width=6,
    color='white'
))
fig.show()

## Distance

In [48]:
temp = g.copy()
temp['abs_error'] = temp['avg_dist'] * temp['moe']
fig = px.bar(temp, x='ratio_standardized', y=['avg_dist', 'std_dist'], error_y='abs_error', barmode='group', color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Distance to Center by Aspect Ratio"))
fig.update_traces(error_y=dict(
    thickness=2,
    width=6,
    color='white'
))
fig.show()

## Gini

In [49]:
temp = g.copy()
temp['abs_error'] = temp['gini'] * temp['moe']
fig = px.bar(temp, 
             x='ratio_standardized', 
             y=['gini'], 
             error_y='abs_error', 
             barmode='group', 
             color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Gini Score by Aspect Ratio"))
fig.update_traces(error_y=dict(
    thickness=2,
    width=6,
    color='white'
))
fig.show()

## Dispersion

In [50]:
temp = g.copy()
temp['abs_error'] = temp['avg_disp'] * temp['moe']
fig = px.bar(temp, 
             x='ratio_standardized', 
             y=['avg_disp'], 
             error_y='abs_error', 
             barmode='group', 
             color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Dispersion by Aspect Ratio"))
fig.update_traces(error_y=dict(
    thickness=2,
    width=6,
    color='white'
))
fig.show()

## Horizontal Spread

In [51]:
temp = g.copy()
temp['abs_error'] = temp['avg_h_spread'] * temp['moe']
fig = px.bar(temp, 
             x='ratio_standardized', 
             y=['avg_h_spread'], 
             error_y='abs_error', 
             barmode='group', 
             color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Horizontal Spread by Aspect Ratio"))
fig.update_traces(error_y=dict(
    thickness=2,
    width=6,
    color='white'
))
fig.show()

## Grid

In [52]:
grid = g[[
    'ratio_standardized',
    'pct_tl',
    'pct_tc',
    'pct_tr',
    'pct_ml',
    'pct_mc',
    'pct_mr',
    'pct_bl',
    'pct_bc',
    'pct_br'
]]
grid
# grid = grid.values.reshape(3, 3)
# grid_norm = grid / grid.sum()
# (grid_norm * 100).round(1)

,ratio_standardized,pct_tl,pct_tc,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br
0,1.33,0.116,0.276,0.114,0.111,0.250,0.115,0.005,0.006,0.006
1,1.66,0.101,0.246,0.098,0.122,0.289,0.121,0.007,0.008,0.008
2,1.85,0.105,0.237,0.101,0.126,0.282,0.127,0.007,0.008,0.008
3,2.35,0.107,0.229,0.098,0.138,0.273,0.129,0.009,0.009,0.009
4,2.55,0.105,0.221,0.097,0.143,0.271,0.137,0.009,0.008,0.009


### 1.33

In [53]:
flat = grid[grid['ratio_standardized'] == '1.33'].drop('ratio_standardized', axis=1)
flat_grid = flat.values.reshape(3, 3)
flat_grid_norm = flat_grid / flat_grid.sum()
(flat_grid_norm * 100).round(1)

array([[11.6, 27.6, 11.4],
       [11.1, 25. , 11.5],
       [ 0.5,  0.6,  0.6]])

In [54]:
utils.plot_grid((flat_grid_norm * 100).round(1), layout=layout)

### 1.85

In [55]:
wide = grid[grid['ratio_standardized'] == '1.85'].drop('ratio_standardized', axis=1)
wide_grid = wide.values.reshape(3, 3)
wide_grid_norm = wide_grid / wide_grid.sum()

In [56]:
utils.plot_grid((wide_grid_norm * 100).round(1), layout=layout)

### 2.35

In [57]:
wide = grid[grid['ratio_standardized'] == '2.35'].drop('ratio_standardized', axis=1)
wide_grid = wide.values.reshape(3, 3)
wide_grid_norm = wide_grid / wide_grid.sum()

In [58]:
utils.plot_grid((wide_grid_norm * 100).round(1), layout=layout)

# Processes

In [59]:
query = """
    SELECT 
        fw.*,
        dw.title,
        dw.year,
        dp.process_name
    FROM factWork fw
    INNER JOIN dimWork dw ON fw.work_id = dw.work_id
    INNER JOIN bridgeProcess as bp ON dw.work_id = bp.work_id
    INNER JOIN dimProcess AS dp ON bp.process_id = dp.process_id
    WHERE dw.year >= 1948
        AND dw.year <= 1958
        AND dp.process_name IN ('CinemaScope', 'Todd-AO', 'VistaVision', 'Spherical')
"""
process_df = pd.read_sql(query, engine).round(3)
process_df

,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,min_size,...,z_pct_face_tv_g,z_v_pct_face_tv,z_v_pct_face_tv_g,z_pct_top1_tv,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g,title,year,process_name
0,2236,33996,34272,0.015,0.036,0.018,0.036,0.065,0.064,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Panhandle,1948,Spherical
1,2844,39188,34812,0.043,0.080,0.053,0.080,0.181,0.118,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bill and Coo,1948,Spherical
2,2848,39195,34816,0.023,0.090,0.035,0.091,0.299,0.299,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Blanche Fury,1948,Spherical
3,2856,39220,34823,0.030,0.067,0.044,0.070,0.414,0.387,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Brighton Rock,1948,Spherical
4,2870,39304,34836,0.028,0.064,0.035,0.064,0.235,0.144,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Daybreak,1948,Spherical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1801,8163,174866,38855,0.016,0.068,0.019,0.068,0.267,0.267,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Let's Live Again,1948,Spherical
1802,8165,175514,54247,0.024,0.111,0.042,0.111,0.359,0.286,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The Case of Charles Peace,1949,Spherical
1803,8177,177782,38865,0.023,0.060,0.031,0.060,0.444,0.444,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fly Away Peter,1948,Spherical
1804,8179,178372,38867,0.017,0.061,0.023,0.061,0.163,0.135,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"David Harding, Counterspy",1950,Spherical


In [60]:
g = process_df[[
    'process_name',
    'avg_size',
    'std_size',
    'avg_f_per_fr',
    'std_f_per_fr',
    'avg_dist',
    'std_dist',
    'gini',
    'avg_disp',
    'std_disp',
    'avg_h_spread',
    'std_h_spread',
    'avg_v_disc',
    'std_v_disc',
    'pct_tl',
    'pct_tc',
    'pct_tr',
    'pct_ml',
    'pct_mc',
    'pct_mr',
    'pct_bl',
    'pct_bc',
    'pct_br'
]].groupby('process_name').mean()
g = g.reset_index()
g = g[g['process_name'].isin(["CinemaScope", "VistaVision", "Todd-AO", "Spherical"])].round(4)
g

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,std_v_disc,pct_tl,pct_tc,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br
0,CinemaScope,0.0137,0.0149,2.4486,2.0320,0.3726,0.1534,0.4802,0.2205,0.1196,...,0.0716,0.1057,0.2258,0.0981,0.1398,0.2728,0.1320,0.0086,0.0084,0.0087
1,Spherical,0.0212,0.0250,2.0047,1.3464,0.3669,0.1426,0.4975,0.2042,0.1079,...,0.0692,0.1142,0.2817,0.1121,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054
2,Todd-AO,0.0093,0.0130,3.7483,4.0460,0.3910,0.1783,0.4360,0.2403,0.1340,...,0.0667,0.1077,0.2040,0.1063,0.1543,0.2673,0.1310,0.0107,0.0063,0.0117
3,VistaVision,0.0138,0.0160,2.4208,2.0120,0.3756,0.1494,0.4873,0.2105,0.1176,...,0.0680,0.1164,0.2799,0.1125,0.1106,0.2487,0.1140,0.0059,0.0058,0.0062


In [61]:
query = """
    SELECT  
        tb.*,
        p.name
    FROM title_basics tb
    INNER JOIN title_process tp ON tb.tconst = tp.tconst
    INNER JOIN process p ON p.id = tp.process_id
    WHERE p.name IN ('CinemaScope', 'Spherical', 'VistaVision', 'Todd-AO')
        AND startYear >= 1948
        AND startYear <= 1958
"""

population_process_df = pd.read_sql(query, imdb_engine)
population_process_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,imdb_id,is_major,name
0,tt0039613,movie,Mary Lou,Mary Lou,0,1948,None,65.0,"Music,Romance",39613,1.0,Spherical
1,tt0039658,movie,The Ghost of Rashmon Hall,Night Comes Too Soon,0,1948,None,52.0,Horror,39658,0.0,Spherical
2,tt0039660,movie,Night Unto Night,Night Unto Night,0,1949,None,84.0,"Drama,Romance",39660,1.0,Spherical
3,tt0039691,movie,Love of a Clown - Pagliacci,Pagliacci - Amore tragico,0,1948,None,67.0,"Drama,Music,Musical",39691,0.0,Spherical
4,tt0039742,movie,May God Forgive Me,Que Dios me perdone,0,1948,None,97.0,Drama,39742,NaN,Spherical
...,...,...,...,...,...,...,...,...,...,...,...,...
5010,tt0463290,movie,Booby Trap,Booby Trap,0,1957,None,71.0,"Crime,Drama,Thriller",463290,0.0,Spherical
5011,tt0483468,movie,The White-Haired Girl,Bai mao nü,0,1951,None,111.0,"Drama,Music",483468,NaN,Spherical
5012,tt2132437,movie,Rocky Marciano vs. Archie Moore,Rocky Marciano vs. Archie Moore,0,1955,None,19.0,Sport,2132437,1.0,Spherical
5013,tt2151544,movie,A Hometown in Heart,Maeumui gohyang,0,1949,None,77.0,Drama,2151544,0.0,Spherical


In [62]:
pop_g = population_process_df.groupby('name').count()
pop_g

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,imdb_id,is_major
name,,,,,,,,,,,
CinemaScope,374,374,374,374,374,374,0,372,372,374,371
Spherical,4555,4555,4555,4555,4555,4555,0,4547,4532,4555,4505
Todd-AO,3,3,3,3,3,3,0,3,3,3,3
VistaVision,83,83,83,83,83,83,0,83,83,83,82


## Counts

In [63]:
cnts = process_df[['process_name', 'avg_size']].groupby('process_name').count()
cnts = cnts[cnts.index.isin(["CinemaScope", "VistaVision", "Cinerama", "Panavision", "Todd-AO", "Spherical"])]
cnts = cnts.rename({"avg_size": "count"}, axis=1)
cnts

,count
process_name,
CinemaScope,275
Spherical,1464
Todd-AO,3
VistaVision,64


In [64]:
cnts['pop'] = pop_g['tconst']
cnts['moe'] = cnts.apply(lambda x: utils.calculate_moe(x['count'], x['pop']), axis=1)
cnts

,count,pop,moe
process_name,,,
CinemaScope,275,374,0.030445
Spherical,1464,4555,0.021101
Todd-AO,3,3,0.000000
VistaVision,64,83,0.058967


## Size

In [65]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['avg_size_error'] = g_error['avg_size'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,avg_size_error
0,CinemaScope,0.0137,0.0149,2.4486,2.0320,0.3726,0.1534,0.4802,0.2205,0.1196,...,0.1398,0.2728,0.1320,0.0086,0.0084,0.0087,275,374,0.030445,0.000417
1,Spherical,0.0212,0.0250,2.0047,1.3464,0.3669,0.1426,0.4975,0.2042,0.1079,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.000447
2,Todd-AO,0.0093,0.0130,3.7483,4.0460,0.3910,0.1783,0.4360,0.2403,0.1340,...,0.1543,0.2673,0.1310,0.0107,0.0063,0.0117,3,3,0.000000,0.000000
3,VistaVision,0.0138,0.0160,2.4208,2.0120,0.3756,0.1494,0.4873,0.2105,0.1176,...,0.1106,0.2487,0.1140,0.0059,0.0058,0.0062,64,83,0.058967,0.000814


In [66]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_size', 'std_size'], 
             error_y=g_error['avg_size_error'],
             barmode='group', 
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Size by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

In [67]:
g_year = process_df[['process_name', 'year', 'avg_size']].groupby(['process_name', 'year']).mean()
g_year = g_year.reset_index()
g_year

,process_name,year,avg_size
0,CinemaScope,1953,0.011667
1,CinemaScope,1954,0.009625
2,CinemaScope,1955,0.010348
3,CinemaScope,1956,0.012327
4,CinemaScope,1957,0.018388
5,CinemaScope,1958,0.016212
6,Spherical,1948,0.022545
7,Spherical,1949,0.022500
8,Spherical,1950,0.021827
9,Spherical,1951,0.020838


In [68]:
fig = px.line(g_year, 
             x='year', 
             y='avg_size',
             color='process_name')
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Size by Production Process"))
fig.update_traces(line=dict(
    width=4
))
fig.show()

## Density

In [69]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['avg_f_per_fr_error'] = g_error['avg_f_per_fr'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,avg_f_per_fr_error
0,CinemaScope,0.0137,0.0149,2.4486,2.0320,0.3726,0.1534,0.4802,0.2205,0.1196,...,0.1398,0.2728,0.1320,0.0086,0.0084,0.0087,275,374,0.030445,0.074549
1,Spherical,0.0212,0.0250,2.0047,1.3464,0.3669,0.1426,0.4975,0.2042,0.1079,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.042302
2,Todd-AO,0.0093,0.0130,3.7483,4.0460,0.3910,0.1783,0.4360,0.2403,0.1340,...,0.1543,0.2673,0.1310,0.0107,0.0063,0.0117,3,3,0.000000,0.000000
3,VistaVision,0.0138,0.0160,2.4208,2.0120,0.3756,0.1494,0.4873,0.2105,0.1176,...,0.1106,0.2487,0.1140,0.0059,0.0058,0.0062,64,83,0.058967,0.142746


In [70]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_f_per_fr', 'std_f_per_fr'], 
             barmode='group', 
             error_y=g_error['avg_f_per_fr_error'],
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Faces per Frame by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Distance

In [71]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['avg_dist_error'] = g_error['avg_dist'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,avg_dist_error
0,CinemaScope,0.0137,0.0149,2.4486,2.0320,0.3726,0.1534,0.4802,0.2205,0.1196,...,0.1398,0.2728,0.1320,0.0086,0.0084,0.0087,275,374,0.030445,0.011344
1,Spherical,0.0212,0.0250,2.0047,1.3464,0.3669,0.1426,0.4975,0.2042,0.1079,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.007742
2,Todd-AO,0.0093,0.0130,3.7483,4.0460,0.3910,0.1783,0.4360,0.2403,0.1340,...,0.1543,0.2673,0.1310,0.0107,0.0063,0.0117,3,3,0.000000,0.000000
3,VistaVision,0.0138,0.0160,2.4208,2.0120,0.3756,0.1494,0.4873,0.2105,0.1176,...,0.1106,0.2487,0.1140,0.0059,0.0058,0.0062,64,83,0.058967,0.022148


In [72]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_dist', 'std_dist'],
             error_y=g_error['avg_dist_error'], 
             barmode='group', 
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Distance from Center by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Dispersion

In [73]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['avg_disp_error'] = g_error['avg_disp'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,avg_disp_error
0,CinemaScope,0.0137,0.0149,2.4486,2.0320,0.3726,0.1534,0.4802,0.2205,0.1196,...,0.1398,0.2728,0.1320,0.0086,0.0084,0.0087,275,374,0.030445,0.006713
1,Spherical,0.0212,0.0250,2.0047,1.3464,0.3669,0.1426,0.4975,0.2042,0.1079,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.004309
2,Todd-AO,0.0093,0.0130,3.7483,4.0460,0.3910,0.1783,0.4360,0.2403,0.1340,...,0.1543,0.2673,0.1310,0.0107,0.0063,0.0117,3,3,0.000000,0.000000
3,VistaVision,0.0138,0.0160,2.4208,2.0120,0.3756,0.1494,0.4873,0.2105,0.1176,...,0.1106,0.2487,0.1140,0.0059,0.0058,0.0062,64,83,0.058967,0.012412


In [74]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_disp', 'std_disp'], 
             barmode='group', 
             error_y=g_error['avg_disp_error'], 
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Dispersion by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Gini

In [75]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['gini_error'] = g_error['gini'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,gini_error
0,CinemaScope,0.0137,0.0149,2.4486,2.0320,0.3726,0.1534,0.4802,0.2205,0.1196,...,0.1398,0.2728,0.1320,0.0086,0.0084,0.0087,275,374,0.030445,0.014620
1,Spherical,0.0212,0.0250,2.0047,1.3464,0.3669,0.1426,0.4975,0.2042,0.1079,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.010498
2,Todd-AO,0.0093,0.0130,3.7483,4.0460,0.3910,0.1783,0.4360,0.2403,0.1340,...,0.1543,0.2673,0.1310,0.0107,0.0063,0.0117,3,3,0.000000,0.000000
3,VistaVision,0.0138,0.0160,2.4208,2.0120,0.3756,0.1494,0.4873,0.2105,0.1176,...,0.1106,0.2487,0.1140,0.0059,0.0058,0.0062,64,83,0.058967,0.028734


In [76]:
fig = px.bar(g, 
             x='process_name', 
             y=['gini'], 
             barmode='group', 
             error_y=g_error['gini_error'],
             color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Gini Score by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Horizontal Spread

In [77]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['h_spread_error'] = g_error['avg_h_spread'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,h_spread_error
0,CinemaScope,0.0137,0.0149,2.4486,2.0320,0.3726,0.1534,0.4802,0.2205,0.1196,...,0.1398,0.2728,0.1320,0.0086,0.0084,0.0087,275,374,0.030445,0.004424
1,Spherical,0.0212,0.0250,2.0047,1.3464,0.3669,0.1426,0.4975,0.2042,0.1079,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.002718
2,Todd-AO,0.0093,0.0130,3.7483,4.0460,0.3910,0.1783,0.4360,0.2403,0.1340,...,0.1543,0.2673,0.1310,0.0107,0.0063,0.0117,3,3,0.000000,0.000000
3,VistaVision,0.0138,0.0160,2.4208,2.0120,0.3756,0.1494,0.4873,0.2105,0.1176,...,0.1106,0.2487,0.1140,0.0059,0.0058,0.0062,64,83,0.058967,0.008108


In [78]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_h_spread'], 
             barmode='group', 
             error_y=g_error['h_spread_error'],
             color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Horizontal Spread by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

In [79]:
from scipy import stats

spherical = process_df[process_df['process_name'] == 'Spherical']['avg_h_spread']
cinemascope = process_df[process_df['process_name'] == 'CinemaScope']['avg_h_spread']
vistavision = process_df[process_df['process_name'] == 'VistaVision']['avg_h_spread']

f_stat, p_value = stats.f_oneway(spherical, cinemascope, vistavision)
print(f"F={f_stat:.3f}, p={p_value:.4f}")

F=36.418, p=0.0000


In [80]:
grand_mean = process_df['avg_h_spread'].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 
                 for g in [spherical, cinemascope, vistavision])
ss_total = sum((process_df['avg_h_spread'] - grand_mean)**2)
eta_squared = ss_between / ss_total
print(f"eta-squared: {eta_squared:.3f}")

eta-squared: 0.039


## Vertical Spread

In [81]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['avg_v_disc'] = g_error['avg_v_disc'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe
0,CinemaScope,0.0137,0.0149,2.4486,2.0320,0.3726,0.1534,0.4802,0.2205,0.1196,...,0.0981,0.1398,0.2728,0.1320,0.0086,0.0084,0.0087,275,374,0.030445
1,Spherical,0.0212,0.0250,2.0047,1.3464,0.3669,0.1426,0.4975,0.2042,0.1079,...,0.1121,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101
2,Todd-AO,0.0093,0.0130,3.7483,4.0460,0.3910,0.1783,0.4360,0.2403,0.1340,...,0.1063,0.1543,0.2673,0.1310,0.0107,0.0063,0.0117,3,3,0.000000
3,VistaVision,0.0138,0.0160,2.4208,2.0120,0.3756,0.1494,0.4873,0.2105,0.1176,...,0.1125,0.1106,0.2487,0.1140,0.0059,0.0058,0.0062,64,83,0.058967


In [82]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_v_disc', 'std_v_disc'], 
             error_y=g_error['avg_v_disc'],
             barmode='group', 
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Vertical Spread by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Grid

In [83]:
grid = g[[
    'process_name',
    'pct_tl',
    'pct_tc',
    'pct_tr',
    'pct_ml',
    'pct_mc',
    'pct_mr',
    'pct_bl',
    'pct_bc',
    'pct_br'
]]
grid

,process_name,pct_tl,pct_tc,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br
0,CinemaScope,0.1057,0.2258,0.0981,0.1398,0.2728,0.1320,0.0086,0.0084,0.0087
1,Spherical,0.1142,0.2817,0.1121,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054
2,Todd-AO,0.1077,0.2040,0.1063,0.1543,0.2673,0.1310,0.0107,0.0063,0.0117
3,VistaVision,0.1164,0.2799,0.1125,0.1106,0.2487,0.1140,0.0059,0.0058,0.0062


#### Spherical

In [84]:
spherical = grid[grid['process_name'] == 'Spherical'].drop('process_name', axis=1)
spherical_grid = spherical.values.reshape(3, 3)
spherical_grid_norm = spherical_grid / spherical_grid.sum()
(spherical_grid_norm * 100).round(1)

array([[11.4, 28.2, 11.2],
       [11. , 25.6, 11. ],
       [ 0.5,  0.6,  0.5]])

In [85]:
utils.plot_grid((spherical_grid_norm * 100).round(1), layout=layout, plot_title="Spherical Grid")

#### CinemaScope

In [86]:
scope = grid[grid['process_name'] == 'CinemaScope'].drop('process_name', axis=1)
scope_grid = scope.values.reshape(3, 3)
scope_grid_norm = scope_grid / scope_grid.sum()
(scope_grid_norm * 100).round(1)

array([[10.6, 22.6,  9.8],
       [14. , 27.3, 13.2],
       [ 0.9,  0.8,  0.9]])

In [87]:
utils.plot_grid((scope_grid_norm * 100).round(1), layout=layout, plot_title="CinemaScope Grid")

#### VistaVision

In [88]:
vista = grid[grid['process_name'] == 'VistaVision'].drop('process_name', axis=1)
vista_grid = vista.values.reshape(3, 3)
vista_grid_norm = vista_grid / vista_grid.sum()
(vista_grid_norm * 100).round(1)

array([[11.6, 28. , 11.2],
       [11.1, 24.9, 11.4],
       [ 0.6,  0.6,  0.6]])

In [89]:
utils.plot_grid((vista_grid_norm * 100).round(1), layout=layout)

#### Todd-AO

In [90]:
todd = grid[grid['process_name'] == 'Todd-AO'].drop('process_name', axis=1)
todd_grid = todd.values.reshape(3, 3)
todd_grid_norm = todd_grid / todd_grid.sum()
(todd_grid_norm * 100).round(1)

array([[10.8, 20.4, 10.6],
       [15.4, 26.7, 13.1],
       [ 1.1,  0.6,  1.2]])

In [91]:
utils.plot_grid((todd_grid_norm * 100).round(1), plot_title="Todd-AO Grid", layout=layout)